# ArgRL spike: argumentation over Overcooked (cramped_room)
Pipeline: `predicates → instantiate → attacks → value_filter → tie_break → grounded_extension`

In [1]:
from overcooked_ai_py.mdp.overcooked_mdp import OvercookedGridworld, Recipe
from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv
from overcooked_ai_py.mdp.actions import Action
from overcooked_ai_py.agents.agent import AgentPair, GreedyHumanModel
from overcooked_ai_py.planning.planners import NO_COUNTERS_PARAMS
from dataclasses import dataclass
from typing import Callable
from collections import Counter
import itertools
import copy

## Setup

In [2]:
mdp = OvercookedGridworld.from_layout_name("cramped_room")

# allow the planner to use counters as motion goals (default NO_COUNTERS_PARAMS forbids them -> dist = inf)
mlam_params = NO_COUNTERS_PARAMS.copy()
mlam_params["counter_goals"] = mdp.get_counter_locations()
env = OvercookedEnv.from_mdp(mdp, horizon=400, mlam_params=mlam_params)
env.reset()
mp = env.mp

print(mdp.state_string(env.state))
print(Action.ALL_ACTIONS)
print(env.state.players[0])
print(env.state.players[1])

Computing MotionPlanner
X       X       P       X       X       

O                       ↑1      O       

X       ↑0                      X       

X       D       X       S       X       


[(0, -1), (0, 1), (1, 0), (-1, 0), (0, 0), 'interact']
(1, 2) facing (0, -1) holding None
(3, 1) facing (0, -1) holding None


### Layout conventions (cramped_room)
- `X` counter, `O` onion dispenser, `P` pot, `D` dish dispenser, `S` serving window
- players `0`, `1`; arrows show the direction they face
- positions are `(x, y)` with `y` increasing downwards
- planner distances include the final *interact* step (hand count + 1); `inf` = unreachable

## Predicates

In [3]:
def predicates(state, mdp, mp):
    """
    Convert raw game state into the abstract facts used by the arguments
    """
    n_players = len(state.players)
    facts = {"players": {}, "pots": {}, "demand": {}, "total_demand": 0}

    # Station locations: fixed for the layout, so look them up once (not per player)
    onion_loc = mdp.get_onion_dispenser_locations()
    dish_loc = mdp.get_dish_dispenser_locations()
    pot_loc = mdp.get_pot_locations()
    serve_loc = mdp.get_serving_locations()
    # Empty counters change with the state (things get placed / picked up)
    empty_counter_loc = mdp.get_empty_counter_locations(state)

    for i in range(n_players):
        player = state.players[i]
        facts["players"][i] = {}

        # Position and orientation (kept for debugging; templates should use dist)
        facts["players"][i]["position"] = player.position
        facts["players"][i]["orientation"] = player.orientation

        # Object Holding
        if player.has_object():
            holding = player.get_object().name
        else:
            holding = "nothing"
        facts["players"][i]["holding"] = holding

        # Distances (planner cost includes the final interact; np.inf = unreachable)
        start = player.pos_and_or
        facts["players"][i]["dist"] = {
            "onion": mp.min_cost_to_feature(start, onion_loc),
            "dish": mp.min_cost_to_feature(start, dish_loc),
            "pot": mp.min_cost_to_feature(start, pot_loc),
            "serve": mp.min_cost_to_feature(start, serve_loc),
            "empty_counter": mp.min_cost_to_feature(start, empty_counter_loc),
        }

    # Pots
    LABELS = {
          "empty": "empty",
          "1_items": "partial",
          "2_items": "partial",
          "3_items": "full_idle",
          "cooking": "cooking",
          "ready": "ready",
      }

    pot_states = mdp.get_pot_states(state)          # once, and from the argument
    for key, positions in pot_states.items():
        label = LABELS[key]                         # loud KeyError on unknown keys
        for pos in positions:
            if key == "empty":
                onions = 0
            else:
                onions = len(state.get_object(pos).ingredients)

            if label in ("empty", "partial"):
                demand = Recipe.MAX_NUM_INGREDIENTS - onions
            else:
                demand = 0

            facts["pots"][pos] = {"state": label, "onions": onions}
            facts["demand"][pos] = demand

    # onions already on their way reduce what's still needed
    carried = sum(1 for p in facts["players"].values() if p["holding"] == "onion")
    facts["total_demand"] = max(0, sum(facts["demand"].values()) - carried)

    return facts

In [4]:
env.reset()
predicates(env.state, mdp, mp)

{'players': {0: {'position': (1, 2),
   'orientation': (0, -1),
   'holding': 'nothing',
   'dist': {'onion': 3, 'dish': 2, 'pot': 3, 'serve': 4, 'empty_counter': 2}},
  1: {'position': (3, 1),
   'orientation': (0, -1),
   'holding': 'nothing',
   'dist': {'onion': 2, 'dish': 4, 'pot': 3, 'serve': 2, 'empty_counter': 1}}},
 'pots': {(2, 0): {'state': 'empty', 'onions': 0}},
 'demand': {(2, 0): 3},
 'total_demand': 3}

## Arguments

### Templates (layout-independent; premises read only `facts`)
| | Premise | Conclusion | Value | dist key |
|---|---|---|---|---|
| T1 | holding(i)=nothing ∧ total_demand>0 | FetchOnion | V_prep | onion |
| T2 | holding(i)=onion ∧ pot_state(p)∈{empty, partial} | PutOnionInPot(p) | V_prep | pot |
| T3 | holding(i)=nothing ∧ pot_state(p)=full_idle | StartCooking(p) | V_cook | pot |
| T4 | holding(i)=nothing ∧ pot_state(p)∈{cooking, ready} | FetchDish | V_serve | dish |
| T4' | holding(i)=nothing ∧ pot_state(p)=partial ∧ total_demand=0 ∧ ∀j: holding(j)≠dish | FetchDish (anticipatory) | V_serve | dish |
| T5 | holding(i)=dish ∧ pot_state(p)=ready | PickUpSoup(p) | V_serve | pot |
| T6 | holding(i)=soup | Deliver | V_deliver | serve |
| T7 | holding(i)≠nothing ∧ ¬reachable(i, target) | PlaceOnCounter(x) | *tbd* | empty_counter |

### Predicates (`facts`)
- `holding(i)` ∈ {nothing, onion, dish, soup}
- `pot_state(p)` ∈ {empty, partial, full_idle, cooking, ready}, plus onion count
- `demand[p]` = onions pot p can still take (0 unless empty/partial); `total_demand` = Σ demand − onions being carried
- `dist(i, ·)` to onion / dish / pot / serve / empty_counter

### Value ranking
V_deliver > V_serve > V_cook > V_prep

### Design notes
- Templates say what is *possible* for one player; attacks decide what is *compatible* across players (e.g. two FetchDish is left to R2).
- T4' is restricted to `partial`: at `full_idle` starting the pot matters more than anticipating the dish.
- T7 postponed: on cramped_room everything is reachable (matters on `forced_coordination`).
- Some states have no applicable argument (holding a dish while the pot cooks; holding a spare onion) → Φ = 0 / agent fallback.

In [5]:
@dataclass(frozen=True)
class Argument:
    agent_idx: int
    action_name: str
    target: tuple[int, int] | None
    value_name: str
    distance: float

@dataclass(frozen=True)
class ArgTemplate:
    name: str
    premise: Callable[[dict, int], list]
    value_name: str
    dist_key: str


VALUE_RANK = {
    "V_deliver": 4,
    "V_serve":   3,
    "V_cook":    2,
    "V_prep":    1,
}

In [6]:
# T1  holding(i)=nothing ∧ total_demand>0                 → i: FetchOnion        | V_prep
def fetch_onion(facts, i):
    holding = facts["players"][i]['holding']
    total_demand = facts["total_demand"]
    if holding == "nothing" and total_demand > 0:
        return [None]
    else:
        return []
    
T1 = ArgTemplate(
    name="FetchOnion",
    premise=fetch_onion,
    value_name="V_prep",
    dist_key="onion"
)

# T2  holding(i)=onion   ∧ pot_state(p)∈{empty, partial}   → i: PutOnionInPot(p)  | V_prep
def put_onion_in_pot(facts, i):
    holding = facts["players"][i]['holding']
    pot_locations = []

    if holding == "onion":
        for pos, info in facts["pots"].items():
            if info["state"] in ["empty", "partial"]:
                pot_locations.append(pos)
    
    return pot_locations

T2 = ArgTemplate(
    name="PutOnionInPot",
    premise=put_onion_in_pot,
    value_name="V_prep",
    dist_key="pot"
)
    
# T3  holding(i)=nothing ∧ pot_state(p)=full_idle          → i: StartCooking(p)   | V_cook
def start_cooking(facts, i):
    holding = facts["players"][i]['holding']
    pot_locations = []

    if holding == "nothing":
        for pos, info in facts["pots"].items():
            if info["state"] == "full_idle":
                pot_locations.append(pos)

    return pot_locations

T3 = ArgTemplate(
    name="StartCooking",
    premise=start_cooking,
    value_name="V_cook",
    dist_key="pot"
)

# T4  holding(i)=nothing ∧ pot_state(p)∈{cooking, ready}   → i: FetchDish         | V_serve
def fetch_dish(facts, i):
    holding = facts["players"][i]['holding']
    if holding == "nothing":
        for pos, info in facts["pots"].items():
            if info["state"] in ["cooking", "ready"]:
                return [None]
    
    return []
            
T4 = ArgTemplate(
    name="FetchDish",
    premise=fetch_dish,
    value_name="V_serve",
    dist_key="dish"
)

# T4' holding(i)=nothing ∧ pot_state(p)=partial ∧ total_demand=0 ∧ ∀j: holding(j)≠dish   → i: FetchDish         | V_serve
def fetch_dish_prime(facts, i):
    holding = facts["players"][i]['holding']
    total_demand = facts["total_demand"]
    someone_has_dish = any(p["holding"] == "dish" for p in facts["players"].values())

    if holding == "nothing" and total_demand == 0 and not someone_has_dish:
        for pos, info in facts["pots"].items():
            if info["state"] == "partial":
                return [None]

    return []

T4p = ArgTemplate(
    name="FetchDish",
    premise=fetch_dish_prime,
    value_name="V_serve",
    dist_key="dish"
)


# T5  holding(i)=dish    ∧ pot_state(p)=ready              → i: PickUpSoup(p)     | V_serve
def pick_up_soup(facts, i):
    holding = facts["players"][i]['holding']
    pot_locations = []

    if holding == "dish":
        for pos, info in facts["pots"].items():
            if info["state"] == "ready":
                pot_locations.append(pos)

    return pot_locations

T5 = ArgTemplate(
    name="PickUpSoup",
    premise=pick_up_soup,
    value_name="V_serve",
    dist_key="pot"
)

# T6  holding(i)=soup   → i: Deliver   | V_deliver
def deliver(facts, i):
    holding = facts["players"][i]['holding']
    if holding == "soup":
        return [None]
    else:
        return []
    

T6 = ArgTemplate(
    name="Deliver",
    premise=deliver,
    value_name="V_deliver",
    dist_key="serve"
)


templates = [T1, T2, T3, T4, T4p, T5, T6]

def instantiate(facts, templates):
    """Apply every template to every player; one Argument per target the premise returns."""
    arguments = set()
    for i in facts["players"]:
        for template in templates:
            for target in template.premise(facts, i):
                arguments.add(Argument(agent_idx=i,
                                       action_name=template.name,
                                       target=target,
                                       value_name=template.value_name,
                                       distance=facts["players"][i]["dist"][template.dist_key]))
    return arguments

## Attacks
- **R1** — same player, different arguments: one pair of hands, one job (always mutual).
- **R2** — different players, same action *and* target, only when claimants > `capacity` (demand-aware).
- **Value filter (VAF)** — drop `(a, b)` when `val(b) > val(a)`.
- **Tie-break** — equal-value mutual clash: closer player wins → lower index (convention) → fixed order for one player's own options.

Selection order = values → distance → convention (each layer is an equilibrium-selection rule; keep them separable for ablations).

In [7]:
def capacity(action_name, target, facts):
    """How many players can usefully take this job right now (R2 fires when claimants exceed it)."""
    if action_name == "FetchOnion":
        # onions still needed overall, already net of onions being carried
        return facts["total_demand"]

    if action_name == "PutOnionInPot":
        # how many more onions this pot can take; the claimants ARE the carriers,
        # so use per-pot demand, not total_demand (which already subtracted them)
        return facts["demand"][target]
        
    if action_name in ("StartCooking", "PickUpSoup"):
        # one press starts a pot; one dish takes the soup
        return 1
    
    if action_name == "FetchDish":
        # pots that will need a dish soon, minus dishes already in someone's hands
        pots_needing_dish = sum(
            1 for info in facts["pots"].values()
            if info["state"] in ("cooking", "ready")
            or (info["state"] in ("partial", "full_idle") and facts["total_demand"] == 0)
        )   
        dishes_held = sum(1 for p in facts["players"].values() if p["holding"] == "dish")
        return max(0, pots_needing_dish - dishes_held)
        
    if action_name == "Deliver":
        # each player delivers their own soup: never a shared job
        return float("inf")

    raise ValueError(f"no capacity rule for action {action_name!r}")


def attacks(arguments, facts):
    attack_list = set()
    # how many arguments claim each job (action + target), counted once up front
    claimants = Counter((a.action_name, a.target) for a in arguments)

    for a, b in itertools.permutations(arguments, 2):
        # Check R1: One Player, one job at a time
        if a.agent_idx == b.agent_idx:
            attack_list.add((a,b))
            continue

        # Check R2
        same_job = (a.action_name == b.action_name) and (a.target == b.target)
        if same_job:
            job = (a.action_name, a.target)
            if claimants[job] > capacity(a.action_name, a.target, facts):
                attack_list.add((a,b))
                
        
    return attack_list


def value_filter(attack_set, value_rank=VALUE_RANK):
    """VAF step: an attack fails if the target promotes a strictly higher value than the attacker."""
    return {
        (a, b) for (a, b) in attack_set
        if value_rank[b.value_name] <= value_rank[a.value_name]
    }

def loses_tie(a, b):
    """True if argument a should give way to b in an equal-value mutual clash."""
    if a.distance != b.distance:
        return a.distance > b.distance            # farther gives way
    if a.agent_idx != b.agent_idx:
        return a.agent_idx > b.agent_idx          # convention: lower index wins
    return (a.action_name, str(a.target)) > (b.action_name, str(b.target))   # same player: fixed order
    
def tie_break(attack_set):
    """Turn equal-value mutual attacks into one-way attacks from the winner to the loser."""
    return {    
        (a, b) for (a, b) in attack_set
        if not ((b, a) in attack_set
                and a.value_name == b.value_name
                and loses_tie(a, b))
    }

## Semantics

In [8]:
# Grounded Extensions
def grounded_extension(args, attacks): # attacks are (attacker, target) pairs
    IN = set()
    OUT = set()

    attackers = {a: set() for a in args}

    for attacker, target in attacks:
        attackers[target].add(attacker)

    while True:
        last_in = set(IN)
        last_out = set(OUT)

        for arg in args:
            if set(attackers[arg]).issubset(OUT):
                IN.add(arg)

        for attack in attacks:
            if attack[0] in IN:
                OUT.add(attack[1])

        if last_in == IN and last_out == OUT:
            break

        
    return IN


def pipeline(facts, templates=templates, use_values=True, use_tie_break=True, value_rank=VALUE_RANK):
    """facts -> (arguments, attacks, grounded extension). Flags switch layers off for ablations."""
    args = instantiate(facts, templates)
    # a job nobody needs (e.g. a second dish) gets no argument, even with a single claimant
    args = {a for a in args if capacity(a.action_name, a.target, facts) >= 1}
    att = attacks(args, facts)
    if use_values:
        att = value_filter(att, value_rank)
    if use_tie_break:
        att = tie_break(att)
    return args, att, grounded_extension(args, att)

## Tests
Scenarios are built with `make_facts` from the reset facts, so they don't depend on a particular greedy run.

In [9]:
def make_facts(base, holdings, pot_state, onions, pot=(2, 0)):
    """Copy of `base` with chosen holdings and pot, demand recomputed the same way predicates does."""
    f = copy.deepcopy(base)
    for i, h in enumerate(holdings):
        f["players"][i]["holding"] = h
    f["pots"] = {pot: {"state": pot_state, "onions": onions}}
    f["demand"] = {pot: (Recipe.MAX_NUM_INGREDIENTS - onions) if pot_state in ("empty", "partial") else 0}
    carried = sum(1 for p in f["players"].values() if p["holding"] == "onion")
    f["total_demand"] = max(0, sum(f["demand"].values()) - carried)
    return f
    
def check(name, f, expected_n_attacks, expected_grounded_size=None):
    args = instantiate(f, templates)
    att = attacks(args, f)
    print(f"{name}: {len(args)} args, {len(att)} attacks")
    for a, b in sorted(att, key=lambda x: (x[0].agent_idx, x[0].action_name)):
        print(f"    P{a.agent_idx}:{a.action_name} -> P{b.agent_idx}:{b.action_name}")
    assert len(att) == expected_n_attacks, f"{name}: expected {expected_n_attacks} attacks, got {len(att)}"
    if expected_grounded_size is not None:
        g = grounded_extension(args, att)
        assert len(g) == expected_grounded_size, f"{name}: expected grounded size {expected_grounded_size}, got {g}"
    return args, att

env.reset()
base = predicates(env.state, mdp, mp)

In [10]:
# grounded semantics on toy frameworks
assert grounded_extension({"a"}, set()) == {"a"}
assert grounded_extension({"a", "b"}, {("a", "b")}) == {"a"}
assert grounded_extension({"a", "b", "c"}, {("a", "b"), ("b", "c")}) == {"a", "c"}
assert grounded_extension({"a", "b"}, {("a", "b"), ("b", "a")}) == set()
assert grounded_extension({"a", "b", "c"}, {("a", "b"), ("b", "a"), ("b", "c")}) == set()

# 4-cycle from the whiteboard: symmetric cook/serve game -> no advice
toy_args    = {"P0:Cook", "P0:Serve", "P1:Cook", "P1:Serve"}
toy_attacks = {("P0:Cook", "P0:Serve"), ("P0:Serve", "P0:Cook"),   # R1: one agent, two tasks
               ("P1:Cook", "P1:Serve"), ("P1:Serve", "P1:Cook"),
               ("P0:Cook", "P1:Cook"),  ("P1:Cook", "P0:Cook"),    # R2: two agents, same task
               ("P0:Serve", "P1:Serve"), ("P1:Serve", "P0:Serve")}
assert grounded_extension(toy_args, toy_attacks) == set()
print("grounded tests passed")

grounded tests passed


In [11]:
# T4': pot partial(2), one player carrying the last onion, the other empty-handed
f = make_facts(base, ["nothing", "onion"], "partial", 2)
assert fetch_dish_prime(f, 0) == [None]          # anticipate the dish
assert fetch_dish_prime(f, 1) == []              # hands not free
f["players"][1]["holding"] = "dish"
assert fetch_dish_prime(f, 0) == []              # someone already has a dish
assert fetch_dish_prime(make_facts(base, ["nothing", "nothing"], "full_idle", 3), 0) == []   # full_idle: StartCooking instead
print("T4' tests passed")

T4' tests passed


In [12]:
# 1. reset: two FetchOnion, demand 3 -> no clash, both accepted
check("reset", base, 0, expected_grounded_size=2) 

# 2. pot empty, both carrying onions: pot takes 3 -> no clash
check("empty pot, two carriers", make_facts(base, ["onion", "onion"], "empty", 0), 0, expected_grounded_size=2)

# 3. pot partial(2), both carrying: pot takes 1 -> mutual R2, grounded empty
check("partial(2), two carriers", make_facts(base, ["onion", "onion"], "partial", 2), 2, expected_grounded_size=0)

# 4. pot cooking, both empty-handed: 1 dish needed -> mutual R2, grounded empty
check("cooking, two empty hands", make_facts(base, ["nothing", "nothing"], "cooking", 3), 2, expected_grounded_size=0)

# 5. cooking, one already holds a dish: only the other has FetchDish -> nothing to clash with
check("cooking, one holds dish", make_facts(base, ["nothing", "dish"], "cooking", 3), 0)

# 6. R1: one player with two arguments, built by hand
A = Argument(0, "StartCooking", (2, 0), "V_cook", 1)
B = Argument(0, "FetchDish", None, "V_serve", 3)
att = attacks([A, B], base)
assert att == {(A, B), (B, A)}, att
print("R1: mutual attack between one player's two arguments")
    
# 7. capacity guard
try:
    capacity("Teleport", None, base)
    assert False, "expected ValueError"
except ValueError:
    print("capacity: unknown action raises ValueError")

print("all attack tests passed")

reset: 2 args, 0 attacks
empty pot, two carriers: 2 args, 0 attacks
partial(2), two carriers: 2 args, 2 attacks
    P0:PutOnionInPot -> P1:PutOnionInPot
    P1:PutOnionInPot -> P0:PutOnionInPot
cooking, two empty hands: 2 args, 2 attacks
    P0:FetchDish -> P1:FetchDish
    P1:FetchDish -> P0:FetchDish
cooking, one holds dish: 1 args, 0 attacks
R1: mutual attack between one player's two arguments
capacity: unknown action raises ValueError
all attack tests passed


In [13]:
# R1 clash with different values: only one direction survives
A = Argument(0, "StartCooking", (2, 0), "V_cook", 1)
B = Argument(0, "FetchDish", None, "V_serve", 3)
filtered = value_filter(attacks([A, B], base))
assert filtered == {(B, A)}, filtered            # V_serve beats V_cook
assert grounded_extension([A, B], filtered) == {B}

# equal values: attacks untouched, grounded still empty
f = make_facts(base, ["nothing", "nothing"], "cooking", 3)
args = instantiate(f, templates)
att = attacks(args, f)
assert value_filter(att) == att
assert grounded_extension(args, value_filter(att)) == set()
print("value filter tests passed")

value filter tests passed


In [14]:
# 1. two FetchDish (cooking), distances 2 vs 4: closer P0 wins
f = make_facts(base, ["nothing", "nothing"], "cooking", 3)
args = instantiate(f, templates)
att = tie_break(value_filter(attacks(args, f)))
g = grounded_extension(args, att)
assert len(att) == 1 and len(g) == 1, (att, g)
winner = next(iter(g))
assert winner.agent_idx == 0 and winner.action_name == "FetchDish", winner

# 2. two PutOnionInPot (partial 2), equal distance 3: convention -> P0 wins
f = make_facts(base, ["onion", "onion"], "partial", 2)
args = instantiate(f, templates)
g = grounded_extension(args, tie_break(value_filter(attacks(args, f))))
assert [a.agent_idx for a in g] == [0], g

# 3. different values are left to the value filter: nothing for tie_break to do
A = Argument(0, "StartCooking", (2, 0), "V_cook", 1)
B = Argument(0, "FetchDish", None, "V_serve", 3)
vf = value_filter(attacks([A, B], base))
assert tie_break(vf) == vf == {(B, A)}

# 4. no clash: untouched
args = instantiate(base, templates)
assert tie_break(value_filter(attacks(args, base))) == set()
assert len(grounded_extension(args, set())) == 2

print("tie-break tests passed")

tie-break tests passed


In [15]:
# whole pipeline, layer by layer, on the "cooking, two empty hands" clash
f = make_facts(base, ["nothing", "nothing"], "cooking", 3)
assert pipeline(f, use_values=False, use_tie_break=False)[2] == set()   # raw: symmetric clash, no advice
assert pipeline(f, use_tie_break=False)[2] == set()                     # values alone can't split equal values
_, _, g = pipeline(f)                                                    # + tie-break: closer player fetches the dish
assert {(a.agent_idx, a.action_name) for a in g} == {(0, "FetchDish")}

# capacity 0: P1 already holds the dish, so P0 gets no FetchDish argument
f = make_facts(base, ["nothing", "dish"], "cooking", 3)
args, _, g = pipeline(f)
assert not any(a.action_name == "FetchDish" for a in args), args
print("pipeline tests passed")

pipeline tests passed


## Exploration: greedy agents + argument trace
Prints whenever the abstract situation (holdings, pots, demand) changes. `✓` marks arguments in the grounded extension.

In [16]:
def abstract_key(facts):
    """The parts of facts that matter to the arguments (ignores positions/distances)."""
    holdings = tuple(p["holding"] for p in facts["players"].values())
    pots = tuple(sorted((pos, info["state"], info["onions"]) for pos, info in facts["pots"].items()))
    return (holdings, pots, facts["total_demand"])

def show_args(arguments, accepted=frozenset()):
    for a in sorted(arguments, key=lambda a: (a.agent_idx, a.action_name)):
        tgt = "" if a.target is None else f"{a.target}"
        mark = "✓" if a in accepted else " "
        print(f"   {mark} P{a.agent_idx}: {a.action_name}{tgt}  [{a.value_name}, d={a.distance}]")

env.reset()
pair = AgentPair(GreedyHumanModel(env.mlam), GreedyHumanModel(env.mlam))
prev_key = None

for t in range(200):
    (a0, _), (a1, _) = pair.joint_action(env.state)
    state, r, done, info = env.step((a0, a1))
    facts = predicates(state, mdp, mp)

    key = abstract_key(facts)
    if key != prev_key:
        print(f"t={t}  r={r}  holding={key[0]}  pots={key[1]}  demand={key[2]}")
        args, att, accepted = pipeline(facts)
        show_args(args, accepted)
        idle = [i for i in facts["players"] if not any(a.agent_idx == i for a in args)]
        if idle:
            print(f"    (no arguments for: {['P%d' % i for i in idle]})")
        prev_key = key
    if done:
        break

Computing MediumLevelActionManager
t=0  r=0  holding=('nothing', 'nothing')  pots=(((2, 0), 'empty', 0),)  demand=3
   ✓ P0: FetchOnion  [V_prep, d=2]
   ✓ P1: FetchOnion  [V_prep, d=1]
t=1  r=0  holding=('nothing', 'onion')  pots=(((2, 0), 'empty', 0),)  demand=2
   ✓ P0: FetchOnion  [V_prep, d=1]
   ✓ P1: PutOnionInPot(2, 0)  [V_prep, d=3]
t=2  r=0  holding=('onion', 'onion')  pots=(((2, 0), 'empty', 0),)  demand=1
   ✓ P0: PutOnionInPot(2, 0)  [V_prep, d=3]
   ✓ P1: PutOnionInPot(2, 0)  [V_prep, d=2]
t=4  r=0  holding=('onion', 'nothing')  pots=(((2, 0), 'partial', 1),)  demand=1
   ✓ P0: PutOnionInPot(2, 0)  [V_prep, d=3]
   ✓ P1: FetchOnion  [V_prep, d=2]
t=6  r=0  holding=('onion', 'onion')  pots=(((2, 0), 'partial', 1),)  demand=0
   ✓ P0: PutOnionInPot(2, 0)  [V_prep, d=2]
   ✓ P1: PutOnionInPot(2, 0)  [V_prep, d=3]
t=14  r=0  holding=('onion', 'nothing')  pots=(((2, 0), 'partial', 2),)  demand=0
   ✓ P0: PutOnionInPot(2, 0)  [V_prep, d=2]
   ✓ P1: FetchDish  [V_serve, d=3]
t=1

## Agent: rule-follower
`ArgAgent` overrides only the decision: each step it runs the pipeline, takes its accepted argument and maps it to motion goals. Movement, interact and auto-unstuck are inherited from `GreedyHumanModel`. No accepted argument → wait (or greedy, with `fallback="greedy"`).

In [17]:
def pots_for(action_name, mdp, state):
    """Pots where this subtask can be done right now (used when no specific target is given)."""
    pot_states = mdp.get_pot_states(state)
    if action_name == "PutOnionInPot":
        return mdp.get_empty_pots(pot_states) + mdp.get_partially_full_pots(pot_states)
    if action_name == "StartCooking":
        return mdp.get_full_but_not_cooking_pots(pot_states)   # not partial: avoids cooking a short soup
    if action_name == "PickUpSoup":
        return mdp.get_ready_pots(pot_states)
    raise ValueError(f"{action_name!r} is not a pot action")


def motion_goals(action_name, target, state, agent_index, mlam):
    mdp = mlam.mdp
    mplanner = mlam.motion_planner
    player = state.players[agent_index]

    counter_objects = mdp.get_counter_objects_dict(state, list(mdp.terrain_pos_dict["X"]))
    goals = []

    if action_name == "FetchOnion":
        goals = mlam.pickup_onion_actions(counter_objects)
    
    elif action_name == "FetchDish":
        goals = mlam.pickup_dish_actions(counter_objects)

    elif action_name in ("PutOnionInPot", "StartCooking", "PickUpSoup"):
        pots = [target] if target is not None else pots_for(action_name, mdp, state)
        goals = mlam._get_ml_actions_for_positions(pots)

    elif action_name == "Deliver":
        goals = mlam.deliver_soup_actions()

    else:
        raise ValueError(f"no motion mapping for {action_name}")

    # 3. keep reachable goals only
    goals = [g for g in goals if mplanner.is_valid_motion_start_goal_pair(player.pos_and_or, g)]
    
    return goals

class ArgAgent(GreedyHumanModel):
    def __init__(self, mlam, fallback="wait", **kwargs):
        super().__init__(mlam, **kwargs)
        self.fallback = fallback
        self.last_arg = None
        self.n_fallback = 0
        self._goals = None

    def action(self, state):
        self._goals = self.decide(state)
        if self._goals is None:
            self.prev_state = state
            return Action.STAY, {"action_probs": self.a_probs_from_action(Action.STAY)}
        return super().action(state)
    
    def ml_action(self, state):
        return self._goals

    def decide(self, state):
        mdp = self.mlam.mdp
        mplanner = self.mlam.motion_planner
        player = state.players[self.agent_index]

        # 1. facts -> accepted arguments for me
        facts = predicates(state, mdp, mplanner)
        _,_, accepted = pipeline(facts)
        mine = [a for a in accepted if a.agent_idx == self.agent_index]
        assert len(mine) <= 1, f"more than one accepted argument for P{self.agent_index}: {mine}"
        arg = mine[0] if mine else None
        self.last_arg = arg

        # 2. argument -> motion goals
        goals = []
        if arg is not None:
            goals = motion_goals(arg.action_name, arg.target, state, self.agent_index, self.mlam)
        
        # 3. nothing to do -> fallback
        if not goals:
            self.n_fallback += 1
            if self.fallback == "greedy":
                return super().ml_action(state)
            return None   # wait: action() turns this into STAY

        return goals
    


In [18]:
import numpy as np

def run(pair, steps=400, seed=0):
    """Play one episode; return (reward, steps carrying an onion no pot can use)."""
    np.random.seed(seed)
    env.reset()
    total, wasted = 0, 0
    for t in range(steps):
        (a0, _), (a1, _) = pair.joint_action(env.state)
        state, r, done, info = env.step((a0, a1))
        total += r
        f = predicates(state, mdp, mp)
        pot_can_take = any(p["state"] in ("empty", "partial") for p in f["pots"].values())
        if f["total_demand"] == 0 and not pot_can_take:
            wasted += sum(1 for p in f["players"].values() if p["holding"] == "onion")
        if done:
            break
    return total, wasted

for seed in range(3):
    arg_pair = AgentPair(ArgAgent(env.mlam), ArgAgent(env.mlam))
    arg_res = run(arg_pair, seed=seed)
    greedy_res = run(AgentPair(GreedyHumanModel(env.mlam), GreedyHumanModel(env.mlam)), seed=seed)
    mixed_res = run(AgentPair(ArgAgent(env.mlam), GreedyHumanModel(env.mlam)), seed=seed)
    print(f"seed {seed}: arg x2 (reward, wasted) = {arg_res}, fallbacks = {[a.n_fallback for a in arg_pair.agents]}"
          f" | greedy x2 = {greedy_res} | arg + greedy = {mixed_res}")

Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
seed 0: arg x2 (reward, wasted) = (0, 0), fallbacks = [16, 386] | greedy x2 = (180, 251) | arg + greedy = (200, 202)
Computing MediumLevelActionManager
Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
seed 1: arg x2 (reward, wasted) = (0, 0), fallbacks = [16, 386] | greedy x2 = (180, 217) | arg + greedy = (180, 173)
Computing MediumLevelActionManager
Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
Computing MediumLevelActionManager
Computing MotionPlanner
seed 2: arg x2 (reward, wasted) = (0, 0), fallbacks = [16, 386] | greedy x2 = (180, 208) | arg + greedy = (180, 195)


## Watch the agents
Renders every step with the library's `StateVisualizer` and shows a timestep slider. The HUD shows each agent's accepted argument.
P0 = **blue** hat, P1 = **green** hat.

In [19]:
import numpy as np
from overcooked_ai_py.visualization.state_visualizer import StateVisualizer

def describe(agent):
    """Short text for the HUD: the agent's accepted argument, or what it fell back to."""
    a = getattr(agent, "last_arg", None)
    if a is None:
        return "greedy" if not isinstance(agent, ArgAgent) else "(no argument: wait)"
    tgt = "" if a.target is None else f" {a.target}"
    return f"{a.action_name}{tgt} [{a.value_name}]"

def record(pair, steps=100, seed=0):
    """Play an episode and keep every state plus HUD text, for the visualiser."""
    np.random.seed(seed)
    env.reset()
    states, rewards, hud = [env.state], [0], []
    score = 0
    for t in range(steps):
        (a0, _), (a1, _) = pair.joint_action(env.state)
        hud.append(StateVisualizer.default_hud_data(env.state, score=score,
                                                    P0=describe(pair.agents[0]), P1=describe(pair.agents[1])))
        state, r, done, info = env.step((a0, a1))
        score += r
        states.append(state); rewards.append(r)
        if done:
            break
    hud.append(StateVisualizer.default_hud_data(states[-1], score=score, P0="", P1=""))
    return {"ep_states": [states], "ep_rewards": [rewards], "mdp_params": [{"terrain": mdp.terrain_mtx}]}, hud

def show(pair, steps=100, seed=0):
    traj, hud = record(pair, steps, seed)
    return StateVisualizer().display_rendered_trajectory(traj, hud_data=hud, ipython_display=True)

In [20]:
# ArgAgent pair (with fallback="wait" this currently shows the pot deadlock around t≈33)
show(AgentPair(ArgAgent(env.mlam), ArgAgent(env.mlam)), steps=100)

Computing MediumLevelActionManager
Computing MotionPlanner


interactive(children=(IntSlider(value=0, description='timestep'), Output()), _dom_classes=('widget-interact',)…

In [21]:
# greedy pair, for comparison
show(AgentPair(GreedyHumanModel(env.mlam), GreedyHumanModel(env.mlam)), steps=100)

Computing MediumLevelActionManager
Computing MotionPlanner


interactive(children=(IntSlider(value=0, description='timestep'), Output()), _dom_classes=('widget-interact',)…

## Q-Learning

In [22]:
ACTIONS = ["FetchOnion", "FetchDish", "PutOnionInPot", "StartCooking", "PickUpSoup", "Deliver", "Wait"]

def abstract_state(facts, i):
    """
    Return tuple of (my_holding, partners_holdings, ((pot_state, onions), ...), total_demand)
    """

    i_holding = facts['players'][i]["holding"]
    partners_holding = []
    for j in range(len(facts["players"])):
        if j != i:
            partners_holding.append(facts['players'][j]["holding"])

    pot_labels = []

    for pos, info in facts["pots"].items():
        pot_labels.append((info["state"], info['onions']))

    total_demand = facts["total_demand"]

    return (i_holding, tuple(partners_holding), tuple(sorted(pot_labels)), total_demand)


def feasible_actions(facts, i):
    """Subtasks player i can physically do now (holding + pot state only; no strategy). Ordered as ACTIONS."""
    holding = facts["players"][i]["holding"]
    pot_states = [info["state"] for info in facts["pots"].values()]
    feasible = {"Wait"}

    if holding == "nothing":
        feasible |= {"FetchOnion", "FetchDish"}
        if any(st == "full_idle" for st in pot_states):
            feasible.add("StartCooking")   # partial pots excluded on purpose: no short soups

    elif holding == "onion":
        if any(st in ("empty", "partial") for st in pot_states):
            feasible.add("PutOnionInPot")

    elif holding == "dish":
        if any(st == "ready" for st in pot_states):
            feasible.add("PickUpSoup")

    elif holding == "soup":
        feasible.add("Deliver")

    return [a for a in ACTIONS if a in feasible]


In [23]:
class QAgent(GreedyHumanModel):
    """Chooses subtasks itself (random for now); the planner executes them."""

    def __init__(self, mlam, timeout=25, seed=None, **kwargs):
        self.timeout = timeout
        self.rng = np.random.default_rng(seed)
        super().__init__(mlam, **kwargs)          # calls self.reset()

    def reset(self):
        super().reset()
        self.current_subtask = None
        self.step_count = 0
        self._goals = None

    def stay(self, state):
        self.prev_state = state                   # keep greedy's unstuck bookkeeping consistent
        return Action.STAY, {"action_probs": self.a_probs_from_action(Action.STAY)}

    def subtask_finished(self, feasible):
        """Done if it is no longer feasible (success or overtaken), timed out, or a 1-step Wait."""
        if self.current_subtask is None:
            return True
        if self.current_subtask not in feasible:
            return True
        if self.current_subtask == "Wait" and self.step_count >= 1:
            return True
        return self.step_count >= self.timeout

    def choose_subtask(self, facts, feasible):
        return self.rng.choice(feasible)          # Step 9 replaces this with epsilon-greedy over Q

    def action(self, state):
        facts = predicates(state, self.mlam.mdp, self.mlam.motion_planner)
        feasible = feasible_actions(facts, self.agent_index)

        # 1. end the current subtask if it is over, then pick a new one
        if self.subtask_finished(feasible):
            self.current_subtask = str(self.choose_subtask(facts, feasible))
            self.step_count = 0
        self.step_count += 1

        # 2. Wait -> stand still
        if self.current_subtask == "Wait":
            return self.stay(state)

        # 3. subtask -> motion goals; none reachable -> give up on it this step
        self._goals = motion_goals(self.current_subtask, None, state, self.agent_index, self.mlam)
        if not self._goals:
            self.current_subtask = None
            return self.stay(state)

        # 4. planner, interact and auto-unstuck from GreedyHumanModel
        return super().action(state)

    def ml_action(self, state):
          return self._goals